# METADATA in McStasScript
The McStas `METADATA` keyword allows attaching arbitrary named text blocks (JSON, Python code, plain text, etc.) to component instances. This notebook demonstrates how to use the METADATA feature in McStasScript.

In [ ]:
import mcstasscript as ms

instr = ms.McStas_instr("metadata_demo")

## Basic usage

Add a component, then attach a METADATA block with `add_METADATA`.

In [ ]:
origin = instr.add_component("Origin", "Progress_bar", AT=[0, 0, 0])

In [ ]:
origin.add_METADATA("stored", "txt", "Hello from McStasScript")
print("Metadata blocks on Origin:")
for block in origin.metadata_list:
    print(f"  {block.name}: type={block.type}, value={block.value}")

In [ ]:
block = origin.get_METADATA("stored")
print(block.name, block.type, block.value)

## Multiple blocks and types

A component can have multiple METADATA blocks. The `type` field is free-form — it can be `JSON`, `Python`, `txt`, or a quoted MIME type like `"application/json"`.

In [ ]:
origin.add_METADATA("config", "JSON", '{"energy": 0.8, "beamsize": 0.05}')
origin.add_METADATA("function", "Python", "def scale(x):\n    return x * 2")

for b in origin.metadata_list:
    print(f"{b.name:10s}  {b.type:8s}  {b.value[:40]}")

## Writing the instrument

METADATA blocks are written to the `.instr` file when the instrument is saved.

In [ ]:
instr.write_full_instrument()

In [ ]:
with open("metadata_demo.instr") as f:
    content = f.read()

# Show only the lines around METADATA blocks
for i, line in enumerate(content.splitlines()):
    if "METADATA" in line or i in [1, 2, 3] and "METADATA" in content.splitlines()[i-1]:
        print(line)

## File.comp pattern

The `File` component uses METADATA blocks to generate input files at simulation start. The `metadatakey` parameter references a block by `ComponentName:metadataName`.

In [ ]:
writer = instr.add_component("writer", "File")
writer.filename = "\"output.txt\""
writer.metadatakey = "\"Origin:stored\""
writer.keep = 1

When this instrument is run with `mcrun`, the `File` component will write the text `"Hello from McStasScript"` to `output.txt` at simulation start.

## Instrument-level queries

The instrument object provides methods to query METADATA across all components.

In [ ]:
print("Readable metadata:")
instr.show_METADATA()
print("Metadata dictionary:", instr.get_METADATA())

In [ ]:
print("Origin metadata dictionary:", instr.get_METADATA("Origin"))

In [ ]:
print("Type: ", instr.metadata_type("Origin", "config"))
print("Data: ", instr.metadata_data("Origin", "config"))

## Removing METADATA

Use `remove_METADATA` to delete a block by name.

In [ ]:
origin.remove_METADATA("function")
print("Metadata blocks after removal:")
for block in origin.metadata_list:
    print(f"  {block.name}: type={block.type}, value={block.value}")